# Spot Detection Comparison: Standard vs Bayer-Specific

This notebook compares two spot detection approaches:

1. **Standard Approach**: Demosaic → Detect on RGB channels
2. **Bayer Approach**: Detect on raw Bayer channels → Map coordinates to full resolution

The Bayer approach preserves noise independence by avoiding demosaicing interpolation,
but operates on subsampled data (50-75% fewer pixels per channel).

**Key Question**: Does preserving noise independence outweigh the reduced spatial resolution?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import sys
import os
import time

# Add src to path
module_dir = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
sys.path.insert(0, module_dir)

import IOFunctions
import MaskFunctions
import sCMOSFunctions
from SpotDetectionFunctions import SpotDetection_Functions
from BayerSpotDetection import detect_spots_bayer_multichannel
from PlottingBase import PublicationPlotter

## Configuration

In [ ]:
# Test data parameters
data_path = '/media/jbeckwith/Ezra Seagat/test_script/20mW638_10p561_NF_SP_1_MMStack_2-Pos000_000.ome.tif'
frame_index = 5  # Which frame to analyze

# Bayer pattern
pattern = 'RGGB'  # Assumed - verify with camera specs
mosaic_unit = np.array([['R', 'G'], ['G', 'B']])

# Detection parameters
pfa = 1e-4  # Probability of false alarm
sigma = 1.5  # PSF width in pixels
fraction_true = 0.2  # Expected fraction of true spots

# Plotting
marker_size = 8
zoom_size = 50  # pixels

print(f"Configuration:")
print(f"  Data: {os.path.basename(data_path)}")
print(f"  Frame: {frame_index}")
print(f"  Pattern: {pattern}")
print(f"  PFA: {pfa}")
print(f"  Sigma: {sigma}")

## Load Data and Create Calibration Maps

In [ ]:
# Load single frame
io = IOFunctions.IO_Functions()
raw_data = io.read_tiff(data_path, dtype='float32', frame=frame_index)

height, width = raw_data.shape
print(f"Image shape: {height} × {width}")
print(f"Data range: [{raw_data.min():.0f}, {raw_data.max():.0f}] ADU")

# Create default calibration maps (uniform - replace with actual calibration if available)
gain_map = np.ones((height, width), dtype=np.float32)
offset_map = np.zeros((height, width), dtype=np.float32)
rqe = np.ones((height, width), dtype=np.float32)
read_noise = np.ones((height, width), dtype=np.float32) * 1.6  # Typical value
variance = read_noise ** 2

# Create Bayer masks
mask_gen = MaskFunctions.Mask_Functions()
masks = mask_gen.get_masks(height, width, mosaic_unit)

## Method 1: Standard Spot Detection (Demosaic → Detect)

In [ ]:
print("="*60)
print("METHOD 1: STANDARD DETECTION (Demosaic → Detect)")
print("="*60)

# Demosaic using variance-aware method
scmos = sCMOSFunctions.sCMOS_Functions()
demosaiced_image = scmos.variance_aware_malvar_demosaic(
    raw_data,
    variance_map=variance,
    offset_map=offset_map,
    gain=gain_map,
    grayscale=True
)

print(f"Demosaiced image shape: {demosaiced_image.shape}")

# Detect spots on demosaiced image
spot_detector = SpotDetection_Functions()

t_start = time.time()
detected_puncta_standard = spot_detector.detect_puncta_in_image(
    demosaiced_image,
    pfa=pfa,
    sigma=sigma,
    variance=variance,
    fraction_true=fraction_true
)
t_standard = time.time() - t_start

n_standard = len(detected_puncta_standard) if detected_puncta_standard is not None else 0
print(f"\nDetected {n_standard} spots in {t_standard*1000:.1f}ms")
print(f"Coordinates: [row, col] format (numpy indexing)")

## Method 2: Bayer-Specific Detection (Detect on Raw Channels)

In [ ]:
print("="*60)
print("METHOD 2: BAYER-SPECIFIC DETECTION (Raw Channels)")
print("="*60)

t_start = time.time()
detections_by_channel, metadata = detect_spots_bayer_multichannel(
    raw_data,
    spot_detector,
    pattern=pattern,
    pfa=pfa,
    sigma=sigma,
    variance=variance,
    channels=['red', 'green', 'blue'],
    fraction_true=fraction_true
)
t_bayer = time.time() - t_start

# Count total detections
n_bayer = sum(len(dets) for dets in detections_by_channel.values())
n_red = len(detections_by_channel.get('red', []))
n_green = len(detections_by_channel.get('green', []))
n_blue = len(detections_by_channel.get('blue', []))

print(f"\nTotal: {n_bayer} spots in {t_bayer*1000:.1f}ms")
print(f"  Red:   {n_red} spots (25% sampling)")
print(f"  Green: {n_green} spots (50% sampling)")
print(f"  Blue:  {n_blue} spots (25% sampling)")
print(f"\nCoordinates: [frame, y, x, ...] format (full resolution)")

## Comparison Statistics

In [ ]:
print("="*60)
print("COMPARISON")
print("="*60)

print(f"\nDetection counts:")
print(f"  Standard (demosaic): {n_standard} spots")
print(f"  Bayer (raw):         {n_bayer} spots")
print(f"  Difference:          {n_bayer - n_standard} ({(n_bayer-n_standard)/max(n_standard,1)*100:+.1f}%)")

print(f"\nTiming:")
print(f"  Standard: {t_standard*1000:.1f}ms")
print(f"  Bayer:    {t_bayer*1000:.1f}ms")
print(f"  Speedup:  {t_standard/t_bayer:.2f}×")

print(f"\nBayer channel breakdown:")
if n_standard > 0:
    print(f"  Red:   {n_red} ({n_red/n_standard*100:.1f}% of standard)")
    print(f"  Green: {n_green} ({n_green/n_standard*100:.1f}% of standard)")
    print(f"  Blue:  {n_blue} ({n_blue/n_standard*100:.1f}% of standard)")

## Visualization: Full Field Comparison

In [ ]:
# Define channel colors
channel_colors = {'red': 'red', 'green': 'lime', 'blue': 'cyan'}

# Create figure
plotter = PublicationPlotter()
fig, axes = plotter.two_column_plot(nrows=1, ncols=2, height=6)

# Calculate percentiles for consistent display
vmin = np.percentile(demosaiced_image, 1)
vmax = np.percentile(demosaiced_image, 99)

# Left: Standard detection
ax = axes[0]
plotter.create_image_plot(ax, demosaiced_image, vmin=vmin, vmax=vmax, cmap='gray')
if detected_puncta_standard is not None and len(detected_puncta_standard) > 0:
    # detected_puncta stores [row, col] = [y, x], scatter needs (x, y)
    ax.scatter(detected_puncta_standard[:, 1], detected_puncta_standard[:, 0],
              s=marker_size, c='yellow', marker='o', alpha=0.6, linewidths=0.5, edgecolors='black')
plotter.setup_axis(ax, title=f'Standard Detection ({n_standard} spots)',
                  xlabel='X (px)', ylabel='Y (px)', grid=False, equal_aspect=True)

# Right: Bayer detection (color-coded by channel)
ax = axes[1]
plotter.create_image_plot(ax, demosaiced_image, vmin=vmin, vmax=vmax, cmap='gray')
for channel, dets in detections_by_channel.items():
    if len(dets) > 0:
        # Detections are [frame, y, x, ...] - extract y, x
        frame_mask = dets[:, 0] == 0  # Single frame, so frame=0
        frame_dets = dets[frame_mask]
        if len(frame_dets) > 0:
            ax.scatter(frame_dets[:, 2], frame_dets[:, 1],
                      s=marker_size, c=channel_colors[channel], 
                      marker='s', alpha=0.6, linewidths=0.5, edgecolors='white',
                      label=f'{channel.capitalize()} ({len(frame_dets)})')
plotter.setup_axis(ax, title=f'Bayer Detection ({n_bayer} spots)',
                  xlabel='X (px)', ylabel='Y (px)', grid=False, equal_aspect=True)
ax.legend(loc='upper right', framealpha=0.9, fontsize=8)

plt.tight_layout()
plt.show()

## Visualization: Zoomed Comparison

In [ ]:
# Find highest density region for zoom
if detected_puncta_standard is not None and len(detected_puncta_standard) > 0:
    x_std = detected_puncta_standard[:, 1]  # col = x
    y_std = detected_puncta_standard[:, 0]  # row = y
    
    density_hist, x_edges, y_edges = np.histogram2d(x_std, y_std, bins=50)
    max_density_idx = np.unravel_index(np.argmax(density_hist), density_hist.shape)
    
    # Center of zoom region
    center_x = (x_edges[max_density_idx[0]] + x_edges[max_density_idx[0] + 1]) / 2
    center_y = (y_edges[max_density_idx[1]] + y_edges[max_density_idx[1] + 1]) / 2
    
    # Zoom window
    min_x, max_x = int(center_x - zoom_size), int(center_x + zoom_size)
    min_y, max_y = int(center_y - zoom_size), int(center_y + zoom_size)
else:
    # Default to center if no detections
    min_x, max_x = width // 2 - zoom_size, width // 2 + zoom_size
    min_y, max_y = height // 2 - zoom_size, height // 2 + zoom_size

print(f"Zoom region: x=[{min_x}, {max_x}], y=[{min_y}, {max_y}]")

# Create zoomed comparison figure
fig, axes = plotter.two_column_plot(nrows=1, ncols=3, height=6)

# Left: Standard detection (zoomed)
ax = axes[0]
plotter.create_image_plot(ax, demosaiced_image, vmin=vmin, vmax=vmax, cmap='gray')
if detected_puncta_standard is not None and len(detected_puncta_standard) > 0:
    ax.scatter(detected_puncta_standard[:, 1], detected_puncta_standard[:, 0],
              s=marker_size*3, c='yellow', marker='o', alpha=0.7, linewidths=1, edgecolors='black')
ax.set_xlim(min_x, max_x)
ax.set_ylim(max_y, min_y)  # Invert y-axis for image coordinates
plotter.setup_axis(ax, title='Standard (Zoom)',
                  xlabel='X (px)', ylabel='Y (px)', grid=False, equal_aspect=True)
plotter.add_scalebar(ax, pixelsize=69, length_nm=1000, label='1 μm', color='white')

# Middle: Bayer detection (zoomed)
ax = axes[1]
plotter.create_image_plot(ax, demosaiced_image, vmin=vmin, vmax=vmax, cmap='gray')
for channel, dets in detections_by_channel.items():
    if len(dets) > 0:
        frame_mask = dets[:, 0] == 0
        frame_dets = dets[frame_mask]
        if len(frame_dets) > 0:
            ax.scatter(frame_dets[:, 2], frame_dets[:, 1],
                      s=marker_size*3, c=channel_colors[channel],
                      marker='s', alpha=0.7, linewidths=1, edgecolors='white',
                      label=channel.capitalize())
ax.set_xlim(min_x, max_x)
ax.set_ylim(max_y, min_y)
plotter.setup_axis(ax, title='Bayer (Zoom)',
                  xlabel='X (px)', ylabel='Y (px)', grid=False, equal_aspect=True)
ax.legend(loc='upper right', framealpha=0.9, fontsize=8)

# Right: Overlay comparison
ax = axes[2]
plotter.create_image_plot(ax, demosaiced_image, vmin=vmin, vmax=vmax, cmap='gray')
# Standard in yellow circles (larger, transparent)
if detected_puncta_standard is not None and len(detected_puncta_standard) > 0:
    ax.scatter(detected_puncta_standard[:, 1], detected_puncta_standard[:, 0],
              s=marker_size*4, c='none', marker='o', alpha=0.5, 
              linewidths=2, edgecolors='yellow', label='Standard')
# Bayer in colored X markers (smaller, opaque)
for channel, dets in detections_by_channel.items():
    if len(dets) > 0:
        frame_mask = dets[:, 0] == 0
        frame_dets = dets[frame_mask]
        if len(frame_dets) > 0:
            ax.scatter(frame_dets[:, 2], frame_dets[:, 1],
                      s=marker_size*2, c=channel_colors[channel],
                      marker='x', alpha=0.8, linewidths=2,
                      label=f'Bayer {channel}')
ax.set_xlim(min_x, max_x)
ax.set_ylim(max_y, min_y)
plotter.setup_axis(ax, title='Overlay (O=Std, X=Bayer)',
                  xlabel='X (px)', ylabel='Y (px)', grid=False, equal_aspect=True)
ax.legend(loc='upper right', framealpha=0.9, fontsize=7)

plt.tight_layout()
plt.show()

## Analysis

### Expected Observations

1. **Green channel should have smallest difference**
   - 50% pixel sampling (quincunx pattern)
   - Best spatial resolution of the Bayer channels

2. **Red/Blue channels show larger differences**
   - Only 25% pixel sampling (checkerboard pattern)
   - More affected by spatial resolution limits

3. **Bayer approach is typically faster**
   - Processing 50-75% fewer pixels per channel
   - Less computational overhead

### Interpretation

If Bayer detects **fewer spots**:
- Could indicate increased false negatives from reduced spatial resolution
- Two nearby spots may alias to same position in subsampled data
- OR standard method has more false positives from correlated noise

If Bayer detects **more spots**:
- Could indicate better noise handling (preserved independence)
- Standard method may suppress true spots due to demosaicing artifacts

**Ground truth validation needed** to determine which method is more accurate.

## Save Results (Optional)

In [ ]:
# Uncomment to save comparison data
# import pandas as pd

# # Save standard detections
# if detected_puncta_standard is not None and len(detected_puncta_standard) > 0:
#     std_df = pd.DataFrame(detected_puncta_standard, columns=['y', 'x'])
#     std_df.to_csv('standard_detections.csv', index=False)

# # Save Bayer detections by channel
# for channel, dets in detections_by_channel.items():
#     if len(dets) > 0:
#         bayer_df = pd.DataFrame(dets, columns=['frame', 'y', 'x', 'intensity'])
#         bayer_df.to_csv(f'bayer_{channel}_detections.csv', index=False)

print("Results can be saved by uncommenting code above.")